# Attentional Blink for LLMs — local run (combined A×B×H design)

Local notebook for **Windows + RTX 1000 Ada (6 GB VRAM), 32 GB RAM**.
Active model = `Qwen2.5-0.5B-Instruct` (wiring tests; switch to 1.5B for interpretable load effects — 0.5B barely solves `semantic_4`).

**Default design (2026-07-20):** `n_tasks` named tasks (1 passphrase + n_tasks−1 loads) scattered among 15 packets; free report `- Task <NAME>: <result>` with the task count never revealed (**H**); a finite CoT budget `finite_budget` caps only the tokens inside `<Thinking>` — on cap the block is force-closed and the answers finish uncut (**A**); `n_tasks` scales the competition (**B**).

- **conscious** = the passphrase appears in the report line of the passphrase task (binary, `report_correct`)
- **unconscious strength** = joint log-prob of the correct passphrase at the model's own `- Task <NAME>:` answer slot

Both read-outs come from a single trajectory per trial. The old single-T1 lag design is still available (`n_tasks=None`); see `README.md`.


In [ ]:
# One-time install in your venv (PowerShell) — uncomment if not already done:
# pip install torch --index-url https://download.pytorch.org/whl/cu121   # CUDA 12.1 build for the RTX 1000 Ada
# pip install "transformers>=4.44" accelerate pandas matplotlib
# pip install bitsandbytes   # only for the 3B 4-bit option below


In [ ]:
# Make the LLM_Blink package importable. This notebook lives INSIDE LLM_Blink/,
# so the repo root (the folder that CONTAINS LLM_Blink/) must be on sys.path.
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # adjust if you move the notebook
from LLM_Blink import load_model, run_sweep, plot_ab, build_trial, TrialConfig
print("LLM_Blink imported OK")


## 0. GPU check


In [ ]:
import torch
print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))


## 1. Load the model
Finite budgets need the HF backend (assistant-turn prefill) — which is what this notebook uses.


In [ ]:
# --- ACTIVE: Qwen 0.5B (fastest, tiny VRAM; wiring tests only) ---
model, tok = load_model("Qwen/Qwen2.5-0.5B-Instruct")

# --- Light & recommended for 6 GB (fp16 ~3.5 GB): ---
# model, tok = load_model("Qwen/Qwen2.5-1.5B-Instruct")

# --- 3B: does NOT fit in 6 GB in fp16 -> use 4-bit (needs `pip install bitsandbytes`): ---
# model, tok = load_model("Qwen/Qwen2.5-3B-Instruct", load_in_4bit=True)

# --- Force CPU (any size, slower; uses your 32 GB RAM). Set BEFORE loading, ---
# --- and restart the kernel if a GPU model was already loaded: ---
# os.environ["CUDA_VISIBLE_DEVICES"] = ""
# model, tok = load_model("Qwen/Qwen2.5-1.5B-Instruct")


## 2. Inspect one trial (sanity-check the stimulus, no GPU needed)


In [ ]:
tr = build_trial(TrialConfig(
    n_tasks=4,                # tasks per stream, INCL. the passphrase task (1..15)
    naming="non-ordered",     # or "ordered"; add passphrase_last=False for a random rank
    t1_load="semantic_4",     # load-task difficulty
    regime="cot",
    seed=0,
))
print(tr.user)
print("\nPassphrase:", tr.t2_phrase, "| its task:", tr.t2_task_name, "| packet:", tr.t2_abs_index)
print("Tasks:", [(t["name"], t["kind"], t["packet"]) for t in tr.tasks])
# Legacy single-T1 design instead:
# tr = build_trial(TrialConfig(lag=2, t1_load="semantic_4", regime="cot", seed=0))


## 3. Pilot sweep
Default A×B×H grid: `n_tasks` (1, 3, 7) × `finite_budget` (64, 256, 1024 CoT tokens) × loads (trivial, semantic_4), cot regime, non-ordered names → **180 trials** at `n_seeds=10` (the loads axis collapses at n_tasks=1). The stimulus is identical across budgets for a given (n_tasks, load, seed), so budget contrasts are paired. Every trial = stage-1 capped `<Thinking>` + stage-2 answers + one teacher-forced scoring pass.


In [ ]:
df = run_sweep(
    model, tok,
    # the arguments below ARE the defaults — spelled out so you can trim them:
    naming="non-ordered",               # H axis: "ordered" | "non-ordered"
    n_tasks_list=(1, 3, 7),             # B axis (incl. the passphrase task; up to 15)
    finite_budgets=(64, 256, 1024),     # A axis: CoT tokens; None = unlimited single pass
    loads=("trivial", "semantic_4"),
    regimes=("cot",),
    passphrase_last=True,               # False -> passphrase at a random rank
    n_seeds=10,
    # answer_budget=512,      # stage-2 (answers) budget — never rationed by design
    # max_new_tokens=1024,    # single-pass budget (finite_budget=None / direct)
    # temperature=0.0,        # 0.3 / 0.7 / 1.0 to sample; graded score conditions on the sampled prefix
)
df.to_csv("ab_results_qwen0.5b_hab.csv", index=False)
df.head()

# Quick smoke run instead (12 trials):
# df = run_sweep(model, tok, n_tasks_list=(1, 3), finite_budgets=(0, 256), n_seeds=2)


## 4. The capacity figures
Left: read-out vs `n_tasks` (one line per budget). Right: read-out vs `finite_budget` (one line per n_tasks). Elastic-capacity signature = load helps at unlimited budget but hurts under tight budgets.


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 2, figsize=(12, 8), squeeze=False)
for r, measure in enumerate(("t2_mean_logprob", "report_correct")):
    plot_ab(df, measure, x="n_tasks", by="finite_budget", ax=axes[r, 0])
    plot_ab(df, measure, x="finite_budget", by="n_tasks", ax=axes[r, 1])
plt.tight_layout(); plt.show()


## 5. Sanity checks — run BEFORE trusting any curve
See `README.md` -> 'Sanity checks & controls'. Gates: `output_truncated` (stage-2/single-pass budget exhausted), `t2_slot_missing` (no scorable answer slot), `thinking_is_placeholder` (instruction text copied into `<Thinking>`), `t2_echoed_in_cot` (passphrase rehearsed pre-slot → graded score ≈ copy-prob; stratify).
Protocol checks (by design, not errors): `cot_forced_closed` should fall as the budget grows; `cot_tokens_used` shows how much Thinking the model actually wanted. Detection: `n_tasks_reported` vs `n_tasks`, plus `n_hallucinated_tasks` (fabricated task names).


In [ ]:
import pandas as pd
# Coerce boolean/None columns to 0/1 floats so means work regardless of dtype
# (fresh dataframe or CSV round-trip).
def _rate(s):
    return s.map({True: 1.0, False: 0.0, "True": 1.0, "False": 0.0})

print("T1 accuracy (FRACTION of load tasks correct) by load x n_tasks:")
print(df.groupby(["t1_load", "n_tasks"], dropna=False)["t1_correct"].mean().round(3))

cot = df[df.regime == "cot"]
print("\nCoT placeholder-copy rate (cot rows):", round(_rate(cot["thinking_is_placeholder"]).mean(), 3))
print("Answer slot-missing rate:", round(_rate(df["t2_slot_missing"]).mean(), 3))
print("Output-truncated rate:", round(_rate(df["output_truncated"]).mean(), 3))
print("Echo (passphrase in pre-slot CoT) rate:", round(_rate(cot["t2_echoed_in_cot"]).mean(), 3))

print("\nProtocol: forced-close rate & CoT tokens used, by budget:")
print(df.groupby("finite_budget", dropna=False)
        .agg(forced=("cot_forced_closed", lambda s: _rate(s).mean()),
             cot_tokens=("cot_tokens_used", "mean")).round(2))

print("\nDetection: tasks reported vs present, and hallucinated names:")
print(df.groupby(["n_tasks", "finite_budget"], dropna=False)
        .agg(reported=("n_tasks_reported", "mean"),
             hallucinated=("n_hallucinated_tasks", "mean")).round(2))

# ---- Gated dataframe: use df_ok for every capacity-curve analysis ----
_bad = pd.Series(False, index=df.index)
for col in ("output_truncated", "t2_slot_missing"):
    _bad |= _rate(df[col]).fillna(0).astype(bool)
df_ok = df[~_bad].copy()
print(f"\nGated df_ok: kept {len(df_ok)}/{len(df)} trials "
      f"({len(df) - len(df_ok)} excluded by truncation / missing answer slot)")
print("Trials kept per cell (watch for empty/unbalanced cells before averaging):")
print(df_ok.groupby(["t1_load", "n_tasks"], dropna=False).size()
           .unstack("n_tasks", fill_value=0))


In [ ]:
# Same figures, but on the GATED data (df_ok) — this is the interpretable version.
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 2, figsize=(12, 8), squeeze=False)
for r, measure in enumerate(("t2_mean_logprob", "report_correct")):
    plot_ab(df_ok, measure, x="n_tasks", by="finite_budget", ax=axes[r, 0])
    plot_ab(df_ok, measure, x="finite_budget", by="n_tasks", ax=axes[r, 1])
plt.suptitle("Gated: no truncation, answer slot present", fontsize=10)
plt.tight_layout(); plt.show()

# Rehearsal stratification (echo => graded score ~ copy-prob):
print(df_ok[df_ok.regime == "cot"]
      .groupby(["t1_load", "t2_echoed_in_cot"], dropna=False)["t2_total_logprob"]
      .mean().round(2))


### Per-task view + inspect single trials
The `tasks` column holds one JSON record per task (name, kind, packet, rank, reported, response, correct) — explode it for detection curves by rank/position.


In [ ]:
import json
import pandas as pd

tasks_long = df_ok[df_ok["tasks"].notna()].copy()
tasks_long["task"] = tasks_long["tasks"].map(json.loads)
tasks_long = tasks_long.explode("task", ignore_index=True)
tasks_long = pd.concat(
    [tasks_long.drop(columns=["task", "tasks"]),
     pd.json_normalize(tasks_long["task"]).add_prefix("task_")], axis=1)

print("Detection (reported) rate by task kind x budget:")
print(tasks_long.groupby(["task_kind", "finite_budget"], dropna=False)["task_reported"]
                .mean().round(3))
print("\nAccuracy by rank within the stream (lost-in-the-middle check):")
print(tasks_long.groupby(["n_tasks", "task_rank"])["task_correct"].mean().round(3))


def show_trial(row):
    "Rebuild + display one trial: works for both task-design and legacy rows."
    r = df.loc[row] if not isinstance(row, pd.Series) else row
    _true = lambda v: (v is True) or (isinstance(v, str) and v.lower() == "true")
    if pd.notna(r.get("n_tasks")):
        cfg = TrialConfig(n_tasks=int(r.n_tasks), naming=str(r.naming),
                          passphrase_last=_true(r.passphrase_last),
                          t1_load=str(r.t1_load), regime=str(r.regime),
                          t2_words=int(r.t2_words), seed=int(r.seed))
        head = (f"n_tasks={int(r.n_tasks)} | budget={r.finite_budget} | "
                f"load={r.t1_load} | naming={r.naming} | seed={r.seed}")
    else:
        cfg = TrialConfig(lag=int(r.lag), t1_load=str(r.t1_load), regime=str(r.regime),
                          mask=_true(r.mask), n_pre=int(r.n_pre), n_post=int(r.n_post),
                          t2_words=int(r.t2_words), seed=int(r.seed))
        head = f"lag={r.lag} | load={r.t1_load} | regime={r.regime} | seed={r.seed}"
    tr = build_trial(cfg)
    print("=" * 90)
    print(head)
    print(f"correct passphrase = {r.t2_phrase!r} (task {tr.t2_task_name}) | "
          f"report_correct={r.report_correct} | t1_correct={r.t1_correct}")
    gates = [c for c in ("output_truncated", "t2_slot_missing",
                         "thinking_is_placeholder", "t2_echoed_in_cot")
             if c in r.index and _true(r[c])]
    print("quality flags:", ", ".join(gates) if gates else "none",
          f"| protocol: cot_forced_closed={r.get('cot_forced_closed')}, "
          f"cot_tokens_used={r.get('cot_tokens_used')}")
    print("\n----- STIMULUS (what the model saw) -----\n" + tr.user)
    print("\n----- MODEL OUTPUT (the realized trajectory) -----\n" + str(r.raw_output))

show_trial(0)


## Next steps
- Switch the model cell to `Qwen2.5-1.5B-Instruct` for the real run (0.5B: `t1_correct` on `semantic_4` ≈ 0.09 → load not paid).
- Controls: `passphrase_last=False` (random rank), `naming="ordered"` (numbered names), `finite_budgets=(..., None)` (unlimited anchor), `regimes=("cot", "direct")`.
- Legacy comparison run: `run_sweep(model, tok, n_tasks_list=(None,), finite_budgets=(None,), lags=(0,2,4,6,8,10), loads=("none","trivial","semantic_4"), regimes=("cot","direct"))`.
- Second model family (e.g. `google/gemma-2-2b-it`) — single-model effects are often idiosyncratic.
- Ollama runs: legacy design only, and re-score graded columns with `rescore_graded.py`.
